#   Preprocessing

In [25]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.polynomial.polynomial import polyfit, polyval

from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.preprocessing import normalize, StandardScaler

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression

from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

import joblib

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

from sklearn.decomposition import KernelPCA
from scipy.sparse import csc_matrix, eye, diags
from scipy.sparse.linalg import spsolve

from sklearn.preprocessing import OneHotEncoder

In [2]:
def load_raman_mapping(file_path):
    df = pd.read_csv(file_path, sep='\s+', header=0)

    df.columns = ['X', 'Y', 'Wave', 'Intensity']

    spectra = df.pivot_table(
        index=['X', 'Y'], 
        columns='Wave', 
        values='Intensity'
    )

    X_raw = spectra.values
    wavenumbers = spectra.columns.values

    return X_raw, wavenumbers

In [3]:
def zscore_mask_only(X_numpy):
    X_safe = X_numpy.astype(np.float64).copy()
    X_safe = np.nan_to_num(X_safe)

    z_scores = np.abs(stats.zscore(X_safe, axis=1))
    max_z = np.max(z_scores, axis=1)

    threshold = np.percentile(max_z, 95)
    keep_mask = max_z < threshold

    kept_pct = keep_mask.mean() * 100
    print(f"Threshold={threshold:.2f}")

    return keep_mask

In [4]:
def parsing_data(dir_path):
    data_list = []

    for dir_1 in os.listdir(dir_path):
        for dir_2 in os.listdir(f'{dir_path}/{dir_1}'):
            for dir_3 in os.listdir(f'{dir_path}/{dir_1}/{dir_2}'):
                print(f'{dir_3}')
                center, cells = None, None
                for i in dir_3.split('_'):
                    if 'center' in i:
                        center = i
                    if 'cortex' in i or 'striatum' in i or 'cerebellum' in i:
                        cells = i

                X_raw, wavenumbers = load_raman_mapping(f'{dir_path}/{dir_1}/{dir_2}/{dir_3}')

                metadata_df = pd.DataFrame({
                    'center': [center] * X_raw.shape[0],
                    'cells': [cells] * X_raw.shape[0], 
                    'cat': [dir_1] * X_raw.shape[0]
                })

                file_data = pd.concat([pd.DataFrame(X_raw), metadata_df], axis=1)
                data_list.append(file_data)

    return pd.concat(data_list, ignore_index=True)

In [ ]:
data = parsing_data('data')

cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place4_1.txt
cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place4_2.txt
cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place5_1.txt
cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place5_2.txt
cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place6_1.txt
cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place6_2.txt
cortex_control_1group_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place4_1.txt
cortex_control_1group_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place4_2.txt
cortex_control_1group_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place5_1.txt
cortex_control_1group_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place5_2.txt
cortex_control_1group_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place6_1.txt

In [88]:
df = data

In [89]:
label_encoder_cat = LabelEncoder()
df['cat'] = label_encoder_cat.fit_transform(data['cat'])

In [90]:
ohe = OneHotEncoder(sparse_output=False, drop='first')
cat_encoded = ohe.fit_transform(df[['center', 'cells']])

In [91]:
df = pd.concat([df.drop(['center', 'cells'], axis=1), pd.DataFrame(cat_encoded, columns=['center', 'cell1', 'cell2'])], axis=1)

In [92]:
X, y = df.drop('cat', axis=1), df['cat']

In [94]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.drop(['center', 'cell1', 'cell2'], axis=1))

In [96]:
X_normalized = normalize(X_scaled, norm='l2')

In [ ]:
pca = PCA(n_components=97)
X_pca = pca.fit_transform(X_normalized)

In [98]:
keep_mask = zscore_mask_only(X_pca)

Threshold=14.03


In [99]:
X = pd.concat([pd.DataFrame(X_pca), X[['center', 'cell1', 'cell2']]], axis=1)

In [103]:
names_columns = [str(i) for i in range(X.shape[1] - 3)]
names_columns.extend(['center', 'cell1', 'cell2'])

X = pd.DataFrame(X.values, columns=names_columns)[keep_mask]
y = y[keep_mask]

In [104]:
X

,0,1,2,3,4,5,6,7,8,9,...,191,192,193,194,195,196,197,center,cell1,cell2
0,1.048557,0.027307,-0.105876,-0.003799,-0.001013,0.000406,0.001841,0.005562,0.001758,-0.004161,...,-0.000609,-0.000306,-0.000091,-0.000087,-0.001360,0.001130,0.000655,0.0,1.0,0.0
1,1.047804,-0.009233,-0.106808,-0.012685,-0.007079,-0.002938,0.001411,0.001331,0.003968,-0.001165,...,0.000579,-0.000265,0.000390,-0.000175,-0.001737,0.000309,-0.000948,0.0,1.0,0.0
2,1.044869,-0.001503,-0.131978,-0.013844,-0.005319,-0.004477,-0.000527,0.003438,0.004006,-0.003749,...,-0.000633,0.001624,-0.001522,0.000721,0.000980,0.000932,0.000563,0.0,1.0,0.0
3,1.042474,-0.080012,-0.105116,-0.030627,-0.013511,-0.004466,-0.000012,0.002328,0.005554,-0.001528,...,-0.000026,-0.000189,-0.000875,-0.002492,0.000994,0.002296,0.000997,0.0,1.0,0.0
4,1.011731,-0.245595,-0.018512,-0.047571,0.005853,0.010007,0.003704,0.011532,0.007478,-0.009623,...,0.001909,0.003139,-0.001910,0.003132,0.010028,-0.005908,-0.000747,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124420,-0.941416,0.065071,-0.001455,-0.009333,-0.001992,0.000683,0.002006,-0.002444,-0.002165,0.002182,...,-0.002273,-0.001221,-0.001132,-0.002831,-0.001833,-0.002642,0.005866,1.0,0.0,0.0
124421,-0.944603,0.051625,0.013664,-0.006357,0.000514,0.000245,0.000630,0.000381,0.000581,0.000073,...,0.001009,0.000970,0.000073,-0.001046,0.000463,0.001043,-0.000119,1.0,0.0,0.0
124422,-0.943997,0.066424,0.012488,-0.003656,0.000927,0.001722,0.004109,0.003589,0.000705,-0.001152,...,0.001456,-0.000368,-0.001182,0.000549,-0.000129,-0.000763,0.000248,1.0,0.0,0.0
124423,-0.943772,0.069031,-0.017112,-0.005784,0.003394,0.005255,0.003950,0.004713,0.000003,-0.001868,...,-0.000309,-0.001179,0.000881,-0.000130,-0.000764,0.001396,-0.000491,1.0,0.0,0.0


In [105]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape[0]}')
print(f'Test:  {X_test.shape[0]}') 

Train: 94562
Test:  23641


In [106]:
rf_model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    verbose=0
)

In [107]:
lgb_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.025,
    max_depth=15,
    num_leaves=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    class_weight='balanced',
    verbose=-1,
    n_jobs=-1
)

In [108]:
xgb_model = xgb.XGBClassifier(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1
)

In [109]:
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)

print(f"RF Accuracy: {accuracy_score(y_test, rf_pred):.3f}")
print(classification_report(y_test, rf_pred, target_names=['сontrol','endo','exo']))

RF Accuracy: 0.848
              precision    recall  f1-score   support

     сontrol       0.84      0.86      0.85      7900
        endo       0.86      0.83      0.85      7219
         exo       0.84      0.85      0.85      8522

    accuracy                           0.85     23641
   macro avg       0.85      0.85      0.85     23641
weighted avg       0.85      0.85      0.85     23641



In [110]:
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)
lgb_proba = lgb_model.predict_proba(X_test)

print(f"LGB Accuracy: {accuracy_score(y_test, lgb_pred):.3f}")
print(classification_report(y_test, lgb_pred, target_names=['сontrol','endo','exo']))

LGB Accuracy: 0.916
              precision    recall  f1-score   support

     сontrol       0.91      0.92      0.91      7900
        endo       0.92      0.92      0.92      7219
         exo       0.91      0.91      0.91      8522

    accuracy                           0.92     23641
   macro avg       0.92      0.92      0.92     23641
weighted avg       0.92      0.92      0.92     23641



In [111]:
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)

print(f"XGB Accuracy: {accuracy_score(y_test, xgb_pred):.3f}")
print(classification_report(y_test, xgb_pred, target_names=['сontrol','endo','exo']))

XGB Accuracy: 0.926
              precision    recall  f1-score   support

     сontrol       0.92      0.92      0.92      7900
        endo       0.94      0.92      0.93      7219
         exo       0.92      0.93      0.92      8522

    accuracy                           0.93     23641
   macro avg       0.93      0.93      0.93     23641
weighted avg       0.93      0.93      0.93     23641



RF Accuracy: 0.848
              precision    recall  f1-score   support

     сontrol       0.84      0.86      0.85      7900
        endo       0.86      0.83      0.85      7219
         exo       0.84      0.85      0.85      8522

    accuracy                           0.85     23641
   macro avg       0.85      0.85      0.85     23641
weighted avg       0.85      0.85      0.85     23641

LGB Accuracy: 0.916
              precision    recall  f1-score   support

     сontrol       0.91      0.92      0.91      7900
        endo       0.92      0.92      0.92      7219
         exo       0.91      0.91      0.91      8522

    accuracy                           0.92     23641
   macro avg       0.92      0.92      0.92     23641
weighted avg       0.92      0.92      0.92     23641

XGB Accuracy: 0.926
              precision    recall  f1-score   support

     сontrol       0.92      0.92      0.92      7900
        endo       0.94      0.92      0.93      7219
         exo       0.92      0.93      0.92      8522

    accuracy                           0.93     23641
   macro avg       0.93      0.93      0.93     23641
weighted avg       0.93      0.93      0.93     23641